In [0]:
from pyspark.sql.functions import (
    col, 
    sum as _sum, 
    count as _count, 
    avg, 
    max as _max, 
    coalesce, 
    lit, 
    current_timestamp
)
from pyspark.sql.types import DecimalType

CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "gold"
TABLE_NAME = "customer_360"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"
GOLD_PATH = f"abfss://gold@bankingdelakevishal.dfs.core.windows.net/{TABLE_NAME}/"

In [0]:
# ============================================
# GOLD: CUSTOMER 360
# ============================================



# 1. Read Silver Tables
dim_customer = spark.table(f"`{CATALOG_NAME}`.silver.customer")
dim_account = spark.table(f"`{CATALOG_NAME}`.silver.account")
dim_loan = spark.table(f"`{CATALOG_NAME}`.silver.loan")
dim_fd = spark.table(f"`{CATALOG_NAME}`.silver.fixed_deposit")
fact_tx = spark.table(f"`{CATALOG_NAME}`.silver.transaction")

# 2. Account Aggregations per Customer
cust_account_agg = (
    dim_account
    .groupBy("customer_id")
    .agg(
        _count("account_id").alias("total_accounts"),
        _sum(col("balance")).alias("total_account_balance"),
        _sum(col("balance")).cast(DecimalType(18, 2)).alias("total_deposit_amount")
    )
)

# 3. Loan Aggregations per Customer
cust_loan_agg = (
    dim_loan
    .filter(col("loan_status") == "ACTIVE")
    .groupBy("customer_id")
    .agg(
        _count("loan_id").alias("active_loans_count"),
        _sum(col("principal")).cast(DecimalType(18, 2)).alias("total_outstanding_loan_amount")
    )
)

# 4. FD Aggregations per Customer
cust_fd_agg = (
    dim_fd
    .filter(col("fd_status") == "ACTIVE")
    .groupBy("customer_id")
    .agg(
        _count("fd_id").alias("active_fd_count"),
        _sum(col("deposit_amount")).cast(DecimalType(18, 2)).alias("total_fd_amount")
    )
)

# 5. Transaction Metrics per Customer
cust_tx_agg = (
    fact_tx
    .join(dim_account.select("account_id", "customer_id"), "account_id", "inner")
    .groupBy("customer_id")
    .agg(
        _count("transaction_id").alias("total_transaction_count"),
        _sum(col("amount")).cast(DecimalType(18, 2)).alias("total_transaction_volume"),
        _max("transaction_timestamp").alias("last_transaction_timestamp")
    )
)

# 6. Build Master Customer 360 View
customer_360_df = (
    dim_customer
    .join(cust_account_agg, "customer_id", "left")
    .join(cust_loan_agg, "customer_id", "left")
    .join(cust_fd_agg, "customer_id", "left")
    .join(cust_tx_agg, "customer_id", "left")
    .select(
        dim_customer["customer_id"],
        col("first_name"),
        col("last_name"),
        col("gender"),
        col("date_of_birth"),
        col("email"),
        col("phone"),
        col("city"),
        col("state"),
        col("country"),
        col("risk_category"),
        col("kyc_status"),
        col("customer_status"),
        col("customer_since"),
        coalesce(col("total_accounts"), lit(0)).alias("total_accounts"),
        coalesce(col("total_account_balance"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_account_balance"),
        coalesce(col("active_loans_count"), lit(0)).alias("active_loans_count"),
        coalesce(col("total_outstanding_loan_amount"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_outstanding_loan_amount"),
        coalesce(col("active_fd_count"), lit(0)).alias("active_fd_count"),
        coalesce(col("total_fd_amount"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_fd_amount"),
        coalesce(col("total_transaction_count"), lit(0)).alias("total_transaction_count"),
        coalesce(col("total_transaction_volume"), lit(0.00)).cast(DecimalType(18, 2)).alias("total_transaction_volume"),
        col("last_transaction_timestamp"),
        current_timestamp().alias("gold_ingestion_timestamp")
    )
)

# 7. Write to ADLS & Unity Catalog
customer_360_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
customer_360_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FULL_TABLE_NAME)

print(f"Customer 360 Gold Table Created: {customer_360_df.count()} records written to '{FULL_TABLE_NAME}'.")

Customer 360 Gold Table Created: 96792 records written to '`banking_lakehouse_db2`.gold.customer_360'.
